# Model Analysis - Exness FX D1

This notebook reads the complete registered validation-prediction population of the bot
`exness_fx_d1` (`bots/exness_fx_d1/BOT.md`). It compares model families, checkpoints, fold
stability, prediction agreement, and uncertainty without choosing a model for deployment. Every
exact model configuration continues to the equal-weight backtest; validation backtest Sharpe
performs selection later. Nothing here reads the holdout: the catalog is filtered to
`split == "validation"` and the holdout stages are the only ones that ever score the sealed
window.

**Learning objectives**

- Audit the complete prediction population by label, family, configuration, and checkpoint.
- Compare predictive diagnostics without using them as selection criteria.
- Evaluate stability and chronological conformal coverage from canonical prediction artifacts.
- Keep causal estimates separate from predictive-family evidence.

**Book reference**: Chapters 11-15

**Prerequisites**: `06_linear` and `07_gbm`. The template's `08_tabular_dl`, `09_dl_tcn`,
`10_dl_nlinear`, `10a_dl_lstm` and `11_causal_dml` are not yet copied into this case study
(phase 4 of the bot fits the linear baseline and the GBM grid); the menu still declares their
members, and the coverage check below names them as declared exclusions rather than letting
the population read as complete.

In [ ]:
"""Analyze the complete exness_fx_d1 validation-prediction population."""

import plotly.express as px
import polars as pl
import yaml
from IPython.display import display
from ml4t.diagnostic.metrics import cross_sectional_ic

import utils.style  # noqa: F401
from case_studies.research import CausalResult, Result, open_study, superseded_members
from case_studies.research.results import PredictionResult
from case_studies.utils.conformal import (
    sizing_conformal_lag,
    walk_forward_conformal_coverage,
)
from utils.modeling import load_configs
from utils.paths import get_case_study_dir

In [ ]:
CASE_STUDY = "exness_fx_d1"
N_BUCKETS = 5
# This notebook reads; it fits nothing and registers nothing, so it has no preview form - a
# preview population is not the thing whose assembly it checks. It still takes the pair,
# because WORKSPACE is what lets a run read an isolated registry instead of the published one.
# Study.open(CASE_STUDY) resolved through the repo case directory, which holds a registry only
# where a maintainer worktree has linked one there; anywhere else it read nothing at all.
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None

## Load the canonical population

A downstream-selectable row must use the current identity schema, carry exact coverage and fold
metrics, and have its prediction artifact available. Causal DML is deliberately absent because it
estimates a treatment effect rather than a cross-sectional score.

In [ ]:
case_dir = get_case_study_dir(CASE_STUDY)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
primary_label = setup["labels"]["primary"]
configured_labels = [primary_label, *setup["labels"].get("variants", [])]
study = open_study(CASE_STUDY, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
catalog = study.predictions.table().filter(
    (pl.col("identity_status") == "current")
    & (pl.col("execution_tier") == "canonical")
    & (pl.col("split") == "validation")
)
# `identity_status` names the schema version a row was written under. It says nothing about
# which generation the row's producer still publishes: a model notebook that refits leaves the
# generation it replaced in the registry, complete and current under that column, so the filter
# above carries retired prediction sets into the analysis. The lineage is what answers it, and
# `superseded_members` reads that - the same exclusion `13_backtest` applies before it freezes
# the baseline population, so the analysed catalog and the backtested one describe one set of
# models rather than two.
retired = superseded_members(study, member_kind="prediction")
if retired:
    catalog = catalog.filter(~pl.col("prediction_hash").is_in(list(retired)))

if catalog.is_empty():
    raise RuntimeError("no current canonical validation predictions are registered")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("model analysis requires complete prediction sets")
if catalog.filter(~pl.col("artifact_available")).height:
    raise RuntimeError("model analysis requires every prediction artifact")
if "causal_dml" in set(catalog.get_column("family")):
    raise RuntimeError("causal DML must not enter the predictive catalog")

identity_columns = [
    "label",
    "family",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
]
if catalog.select(identity_columns).n_unique() != catalog.height:
    raise RuntimeError("prediction rows do not retain one complete model identity")

## Compare the assembled population against the configured menu

Each notebook checks that it produced what it requested. Nothing checks that the requests covered
what the case study configures, and a population that is internally consistent but short a
configured model reads exactly like a complete one. This compares the assembled population to the
menu itself.

The menus were copied from `fx_pairs` whole, so they declare `tabular_dl` and `deep_learning`
members that no notebook in this case study produces yet. Those members are excluded by family
and reason rather than passing unnoticed, and the exclusions are derived from the menu per label
rather than listed, so a label that never declares a member does not gain a phantom exclusion
for it. `FITTED_FAMILIES` is the set of families whose stage exists here; when `08`-`10a` are
copied, extend it and the check below starts requiring their members.

In [ ]:
FITTED_FAMILIES = {"linear", "gbm"}
EXCLUSION_REASON = (
    "declared in the menu copied from fx_pairs, but its stage is not yet copied into "
    "exness_fx_d1 (bots/exness_fx_d1/BOT.md, phase 4 fits linear and gbm)"
)

configured_members = {
    (label, family, config["config_name"])
    for label in configured_labels
    for family in ("linear", "gbm", "tabular_dl", "deep_learning")
    for config in load_configs(CASE_STUDY, label, family=family)
}
excluded_members = {
    (label, family, config_name)
    for label, family, config_name in configured_members
    if family not in FITTED_FAMILIES
}
expected_members = configured_members - excluded_members
present_members = set(catalog.select("label", "family", "config_name").unique().iter_rows())

if excluded_members:
    print(f"Declared exclusions ({EXCLUSION_REASON}):")
    for member in sorted(excluded_members):
        print(f"  {member[0]} / {member[1]} / {member[2]}")

missing_members = sorted(expected_members - present_members)
unexpected_members = sorted(present_members - expected_members)
if missing_members or unexpected_members:
    raise RuntimeError(
        "the assembled population does not match the configured menu; "
        f"missing {missing_members}, unexpected {unexpected_members}"
    )
if set(catalog.get_column("label")) != set(configured_labels):
    raise RuntimeError("the canonical prediction population does not cover every configured label")

In [ ]:
population_summary = (
    catalog.group_by("label", "family")
    .agg(
        pl.col("config_name").n_unique().alias("configurations"),
        pl.len().alias("checkpoints"),
        pl.col("complete").all().alias("complete"),
        pl.col("ic_mean").is_not_null().sum().alias("diagnosed_checkpoints"),
    )
    .sort("label", "family")
)
population_summary

## Compare predictive diagnostics

Rank correlation is computed across the five pairs at each decision date and then averaged
through time. The rows shown below are descriptive representatives for plots. They do not filter
the official prediction population and do not choose checkpoints for backtesting.

In [ ]:
diagnosed = catalog.filter(pl.col("ic_mean").is_not_null())
representatives = (
    diagnosed.sort(
        ["label", "family", "ic_mean", "config_name", "checkpoint_value", "prediction_hash"],
        descending=[False, False, True, False, False, False],
        nulls_last=True,
    )
    .group_by("label", "family", maintain_order=True)
    .first()
    .sort("label", "family")
)
representatives.select(
    "label",
    "family",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "ic_mean",
    "ic_t",
    "prediction_hash",
)

In [ ]:
# Families whose checkpoint is `final` carry no numeric checkpoint position, so they have no x
# coordinate on this axis and plotly would drop them without saying so. They are excluded here by
# name rather than silently, and the count is stated in the title.
numbered = diagnosed.filter(pl.col("checkpoint_value").is_not_null())
unnumbered = diagnosed.filter(pl.col("checkpoint_value").is_null())
excluded_families = sorted(set(unnumbered.get_column("family"))) if unnumbered.height else []
figure = px.scatter(
    numbered,
    x="checkpoint_value",
    y="ic_mean",
    color="family",
    facet_row="label",
    hover_data=["config_name", "prediction_hash"],
    title=(
        "Validation rank correlation across numbered model checkpoints"
        + (
            f" (excludes {unnumbered.height} rows from {', '.join(excluded_families)},"
            " whose checkpoint is `final` and has no numeric position)"
            if excluded_families
            else ""
        )
    ),
    labels={"checkpoint_value": "Checkpoint", "ic_mean": "Mean daily rank correlation"},
)
figure.show()

## Load canonical prediction columns

Producers use `symbol`, `timestamp`, `fold`, `prediction`, and `actual`. Adding the full catalog
identity to each frame keeps two checkpoints from collapsing into one plot label or fold count.

In [ ]:
prediction_frames = []
for row in representatives.iter_rows(named=True):
    result = Result.open(study, row["prediction_hash"])
    if not isinstance(result, PredictionResult):
        raise TypeError(f"{row['prediction_hash']} is not a prediction result")
    frame = result.load()
    required = {"symbol", "timestamp", "fold", "prediction", "actual"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"prediction {result.hash} lacks canonical columns: {sorted(missing)}")
    if frame.select("symbol", "timestamp", "fold").n_unique() != frame.height:
        raise ValueError(f"prediction {result.hash} has duplicate eligible keys")
    prediction_frames.append(
        frame.with_columns(
            pl.lit(row["label"]).alias("label"),
            pl.lit(row["family"]).alias("family"),
            pl.lit(row["config_name"]).alias("config_name"),
            pl.lit(row["checkpoint_value"]).alias("checkpoint_value"),
            pl.lit(row["prediction_hash"]).alias("prediction_hash"),
        )
    )

representative_predictions = pl.concat(prediction_frames, how="diagonal_relaxed")

## Fold stability

Fold identifiers are labels, not dates. The table orders validation windows by their earliest
timestamp and retains checkpoint identity in every row.

In [ ]:
fold_rows = []
for keys, frame in representative_predictions.group_by(
    "label", "family", "config_name", "checkpoint_value", "prediction_hash", "fold"
):
    stats = cross_sectional_ic(
        frame.select("symbol", "timestamp", "prediction"),
        frame.select("symbol", "timestamp", "actual"),
        pred_col="prediction",
        ret_col="actual",
        date_col="timestamp",
        entity_col="symbol",
        min_obs=4,
    )
    fold_rows.append(
        {
            "label": keys[0],
            "family": keys[1],
            "config_name": keys[2],
            "checkpoint_value": keys[3],
            "prediction_hash": keys[4],
            "fold": keys[5],
            "validation_start": frame.get_column("timestamp").min(),
            "ic_mean": stats["ic_mean"],
            "n_decision_times": stats["n_periods"],
        }
    )
fold_metrics = pl.DataFrame(fold_rows).sort("label", "validation_start", "family")
fold_metrics

In [ ]:
fold_figure = px.box(
    fold_metrics,
    x="family",
    y="ic_mean",
    color="family",
    facet_row="label",
    points="all",
    title="Validation rank correlation varies across chronological folds",
    labels={"family": "Model family", "ic_mean": "Fold mean rank correlation"},
)
fold_figure.show()

## Prediction agreement and ranking shape

Correlation between model scores shows whether families rank pairs similarly. One correlation
matrix covers one label: scores fitted against different targets are not comparable ranks of the
same quantity, so the pivot takes the primary label the menu declares rather than all three.
Return buckets and conformal coverage carry no such restriction and run over every label.

In [ ]:
primary = representative_predictions.filter(pl.col("label") == primary_label)
print(f"Score agreement is computed on {primary_label}, the primary label in the menu")
# The pivot index is the canonical eligibility key alone. `actual` is the same realized return for
# every model at a given key, but the families do not agree on its float representation, so including
# it split the rows into disjoint groups - each score column populated on a different half - and every
# correlation, including the diagonal, came out NaN.
wide = primary.pivot(
    on="prediction_hash",
    index=["symbol", "timestamp", "fold"],
    values="prediction",
)
prediction_columns = [
    column for column in wide.columns if column not in {"symbol", "timestamp", "fold"}
]
if wide.height != primary.height // primary.get_column("prediction_hash").n_unique():
    raise RuntimeError("the agreement pivot did not align every representative on the same keys")
if any(wide.get_column(column).null_count() for column in prediction_columns):
    raise RuntimeError("a representative is missing scores at keys the others cover")
agreement = wide.select(prediction_columns).corr()
agreement

In [ ]:
bucket_rows = []
for keys, frame in representative_predictions.group_by(
    "label", "family", "config_name", "checkpoint_value", "prediction_hash", "timestamp"
):
    if frame.height < N_BUCKETS:
        continue
    ranked = frame.sort("prediction").with_columns(
        ((pl.int_range(pl.len()) * N_BUCKETS) // pl.len())
        .clip(upper_bound=N_BUCKETS - 1)
        .alias("bucket")
    )
    for bucket, values in ranked.group_by("bucket"):
        bucket_rows.append(
            {
                "label": keys[0],
                "family": keys[1],
                "config_name": keys[2],
                "checkpoint_value": keys[3],
                "prediction_hash": keys[4],
                "timestamp": keys[5],
                "bucket": bucket[0],
                "actual": values.get_column("actual").mean(),
            }
        )
bucket_summary = (
    pl.DataFrame(bucket_rows)
    .group_by("label", "family", "config_name", "checkpoint_value", "prediction_hash", "bucket")
    .agg(pl.col("actual").mean().alias("mean_realized_return"))
    .sort("label", "family", "bucket")
)
# Three labels sort as 1d, 21d, 5d, so the default row window hides the middle one entirely and
# a reader would see the same two-thirds coverage this section exists to remove.
with pl.Config(tbl_rows=bucket_summary.height):
    display(bucket_summary)

## Coverage of the widths that size positions

The width reported here is the one `conformal_weighted` allocates with: calibrated per symbol on
every residual known at `t - h`, where `h` is the label's horizon in data steps
(`HOLDOUT_CONFORMAL_EMBARGO_STEPS` in `case_studies/utils/conformal.py` holds this case
study's three entries, 1 / 5 / 21 sessions), with a pooled quantile where a symbol has too few
of its own. A decision is covered when its absolute residual falls inside that half-width.

Read it as a diagnostic of residual dispersion, not as a guarantee. Split conformal's
finite-sample coverage needs the calibration and evaluation residuals to be exchangeable, and
currency returns are heteroskedastic and regime-dependent. Nothing in the allocation path reads
an interval or a coverage level - the width stands in for a volatility estimate, and `n_test`
counts the decisions a width could be calibrated for.

In [ ]:
conformal_rows = []
for keys, frame in representative_predictions.group_by(
    "label", "family", "config_name", "checkpoint_value", "prediction_hash"
):
    embargo_steps = sizing_conformal_lag(CASE_STUDY, keys[0])
    for row in walk_forward_conformal_coverage(frame, embargo_steps=embargo_steps):
        conformal_rows.append(
            {
                "label": keys[0],
                "family": keys[1],
                "config_name": keys[2],
                "checkpoint_value": keys[3],
                "prediction_hash": keys[4],
                **row,
            }
        )
conformal = pl.DataFrame(conformal_rows).sort("label", "nominal_level", "family")
with pl.Config(tbl_rows=conformal.height, tbl_cols=conformal.width, tbl_width_chars=200):
    display(conformal)

## Causal evidence remains separate

A causal result answers whether the configured momentum treatment has an estimated effect after
adjustment for its declared confounders. It does not count as predictive-family coverage and does
not enter the model or backtest population. `11_causal_dml` registers one estimate per label the
menu declares, so naming a single label here would leave the rest of them unreported. Until that
stage is copied into this case study the registry holds no causal run, and the frame below says
so per label instead of failing the analysis.

In [ ]:
causal_rows = []
for label in configured_labels:
    try:
        causal = CausalResult.one(study, label=label)
    except ValueError as exc:
        print(f"{label}: no causal estimate registered ({exc}); 11_causal_dml is not yet copied")
        continue
    causal_rows.append({"label": label, "causal_hash": causal.hash, **causal.metrics})
if causal_rows:
    causal_summary = (
        pl.DataFrame(causal_rows)
        .select(
            "label",
            "causal_hash",
            "n_obs",
            "dml_effect",
            "dml_se_hac",
            "p_value_hac",
            "naive_effect",
            "confounding_bias_pct",
            "refutation_p",
        )
        .sort("label")
    )
    display(causal_summary)

## Handoff to the equal-weight backtest

Every complete catalog row, including every checkpoint, advances to `13_backtest`. The backtest
receives these rows directly and records one result per row. Neither this notebook's descriptive
representatives nor any IC or causal statistic changes that population.

In [ ]:
handoff = catalog.select([*identity_columns, "cv_identity", "complete"]).sort(
    "label", "family", "config_name", "checkpoint_value"
)
handoff

## Key takeaways

- Model identity includes label, family, configuration, checkpoint, and validation protocol.
- Daily rank correlation, fold stability, bucket shape, and conformal coverage are diagnostics.
- The equal-weight validation backtest receives the complete prediction population.
- Causal estimates remain a separate form of evidence.